# 02. Preprocessing

**Course:** CSE437 Data Science
**Group:** 15
**Project:** Predicting Google Play Store App Ratings Using Machine Learning

**Purpose of this notebook.** This notebook turns the audited raw dataset
(`01_data_audit_and_eda.ipynb`) into a single clean, analysis-ready table:
`data/processed/googleplaystore_clean.csv`. Every action below is deterministic and
evidence-based — duplicate removal, malformed-row handling, type parsing, target
validity, and outlier *identification* (with a documented decision per variable).

**What this notebook deliberately does NOT do.** No `StandardScaler`, no PCA, no
feature selector, and no *learned* imputation statistic (e.g. a column mean/median
computed from the data) is fit here. Those are train-only, leakage-sensitive
transformations. Per the project's leakage-prevention rule, they belong inside a
`sklearn` `Pipeline`/`ColumnTransformer` fit *after* the train/test split, in
`04_modeling_and_tuning.ipynb`. Where a column has missing values that would
normally need a learned statistic to fill (e.g. `Size_MB` for apps whose size
"Varies with device"), this notebook leaves those values as `NaN` and documents that
they are deferred to the modeling pipeline.


## 0. Imports and Reproducibility Setup

In [1]:
import os
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

RAW_DATA_PATH = "../data/raw/googleplaystore.csv"
PROCESSED_DIR = "../data/processed"
CLEAN_OUTPUT_PATH = f"{PROCESSED_DIR}/googleplaystore_clean.csv"

os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Random seed fixed at:", RANDOM_STATE)


Random seed fixed at: 42


## 1. Load Raw Data (Relative Path)

In [2]:
df = pd.read_csv(RAW_DATA_PATH)
print("Loaded:", RAW_DATA_PATH)
print("Raw shape:", df.shape)

# stage_log records row/column counts and key quality statistics after every
# cleaning stage, for the Section 5 before-vs-after table.
stage_log = []

def log_stage(name, d, note=""):
    stage_log.append({
        "stage": name,
        "rows": d.shape[0],
        "cols": d.shape[1],
        "unique_apps": d["App"].nunique(),
        "duplicate_app_rows": int(d.duplicated(subset=["App"]).sum()),
        "missing_rating": int(d["Rating"].isnull().sum()) if "Rating" in d.columns else np.nan,
        "note": note,
    })

log_stage("0_raw_loaded", df, "Unmodified raw file")
pd.DataFrame(stage_log)


Loaded: ../data/raw/googleplaystore.csv
Raw shape: (10841, 13)


,stage,rows,cols,unique_apps,duplicate_app_rows,missing_rating,note
0,0_raw_loaded,10841,13,9660,1181,1474,Unmodified raw file


## 2. Malformed Shifted Row — Detection and Correction

As established with evidence in `01_data_audit_and_eda.ipynb`, row index `10472`
(`App = "Life Made WI-Fi Touchscreen Photo Frame"`) has every field from `Category`
onward shifted one column to the left, because the true `Category` value was blank
at source. The detection rule is `Rating > 5.0` (structurally impossible on a 1–5
scale), which uniquely identifies this one row.

**Reconstruction.** Reading the shifted values against what each field is *supposed*
to hold produces an internally consistent, high-confidence reconstruction:

| Field | Shifted (raw) value | Reconstructed (true) value |
|---|---|---|
| Category | `'1.9'` | `NaN` (missing at source — this is what caused the shift) |
| Rating | `19.0` | `1.9` |
| Reviews | `'3.0M'` | `19` |
| Size | `'1,000+'` | `'3.0M'` |
| Installs | `'Free'` | `'1,000+'` |
| Type | `'0'` | `'Free'` |
| Price | `'Everyone'` | `'0'` |
| Content Rating | `NaN` | `'Everyone'` |
| Genres | `'February 11, 2018'` | `NaN` |
| Last Updated | `'1.0.19'` | `'February 11, 2018'` |
| Current Ver | `'4.0 and up'` | `'1.0.19'` |
| Android Ver | `NaN` | `'4.0 and up'` |

Every reconstructed value is plausible for its column (a valid 1–5 rating, an
integer review count, an `'M'`-suffixed size, a comma/`+`-formatted install count,
`Free`/`Paid`, a `$`-free price string, a real content-rating tier, a parseable
date, and version strings) — this is not a guess, it is the unique consistent
un-shift. The one field that **cannot** be recovered is the original `Category`,
because that is precisely the value that went missing and caused the shift.


In [3]:
MALFORMED_IDX = 10472
before_shape = df.shape

# Show the row before correction, for the record
print("Row before correction:")
print(df.loc[MALFORMED_IDX])


Row before correction:
App               Life Made WI-Fi Touchscreen Photo Frame
Category                                              1.9
Rating                                               19.0
Reviews                                              3.0M
Size                                               1,000+
Installs                                             Free
Type                                                    0
Price                                            Everyone
Content Rating                                        NaN
Genres                                  February 11, 2018
Last Updated                                       1.0.19
Current Ver                                    4.0 and up
Android Ver                                           NaN
Name: 10472, dtype: object


In [4]:
assert df.loc[MALFORMED_IDX, "App"] == "Life Made WI-Fi Touchscreen Photo Frame"

df.loc[MALFORMED_IDX, "Category"] = np.nan
df.loc[MALFORMED_IDX, "Rating"] = 1.9
df.loc[MALFORMED_IDX, "Reviews"] = "19"
df.loc[MALFORMED_IDX, "Size"] = "3.0M"
df.loc[MALFORMED_IDX, "Installs"] = "1,000+"
df.loc[MALFORMED_IDX, "Type"] = "Free"
df.loc[MALFORMED_IDX, "Price"] = "0"
df.loc[MALFORMED_IDX, "Content Rating"] = "Everyone"
df.loc[MALFORMED_IDX, "Genres"] = np.nan
df.loc[MALFORMED_IDX, "Last Updated"] = "February 11, 2018"
df.loc[MALFORMED_IDX, "Current Ver"] = "1.0.19"
df.loc[MALFORMED_IDX, "Android Ver"] = "4.0 and up"

print("Row after correction:")
print(df.loc[MALFORMED_IDX])


Row after correction:
App               Life Made WI-Fi Touchscreen Photo Frame
Category                                              NaN
Rating                                                1.9
Reviews                                                19
Size                                                 3.0M
Installs                                           1,000+
Type                                                 Free
Price                                                   0
Content Rating                                   Everyone
Genres                                                NaN
Last Updated                            February 11, 2018
Current Ver                                        1.0.19
Android Ver                                    4.0 and up
Name: 10472, dtype: object


**Decision: drop this single row after correction.** `Category` is a core
predictor named explicitly in the approved research questions, and its true value
is genuinely unrecoverable (it was empty at source, which is *why* the shift
happened). Imputing a guessed category for one row would introduce unjustified bias
for negligible benefit. This is **1 row out of 10,841 (0.009%)** — the information
loss is negligible, and the decision is documented here rather than silently
applied.


In [5]:
before = df.shape[0]
df = df[df["Category"].notnull()].copy()
n_dropped = before - df.shape[0]
print(f"Rows dropped (malformed row, unrecoverable Category): {n_dropped}")
print("Shape after malformed-row handling:", df.shape)

log_stage("1_malformed_row_handled", df,
          "1 row dropped: Category unrecoverable after shift correction")
pd.DataFrame(stage_log)


Rows dropped (malformed row, unrecoverable Category): 1
Shape after malformed-row handling: (10840, 13)


,stage,rows,cols,unique_apps,duplicate_app_rows,missing_rating,note
0,0_raw_loaded,10841,13,9660,1181,1474,Unmodified raw file
1,1_malformed_row_handled,10840,13,9659,1181,1474,1 row dropped: Category unrecoverable after sh...


## 3. Duplicate `App` Removal

Per the faculty instruction, duplicates are removed **by `App` name**. Where an app
has multiple rows, they differ mainly in `Reviews` (consistent with re-scraping at
different times); the row with the **highest `Reviews` count** is kept as the most
information-rich / most recent-looking snapshot for that app. `Reviews` is
converted to numeric first (it is already a plain integer string for every
remaining row, since the one non-numeric case was the malformed row handled above).


In [6]:
df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")
print("Non-numeric Reviews remaining after malformed-row fix:", df["Reviews"].isnull().sum())


Non-numeric Reviews remaining after malformed-row fix: 0


In [7]:
before = df.shape[0]
before_unique = df["App"].nunique()

df = (
    df.sort_values("Reviews", ascending=False)
      .drop_duplicates(subset=["App"], keep="first")
      .sort_index()
)

n_removed = before - df.shape[0]
print(f"Rows before dedup:        {before}")
print(f"Unique App names:         {before_unique}")
print(f"Duplicate rows removed:   {n_removed}")
print(f"Rows after dedup:         {df.shape[0]}")
assert df["App"].is_unique

log_stage("2_deduplicated", df, f"{n_removed} duplicate App rows removed (kept max Reviews per App)")
pd.DataFrame(stage_log)


Rows before dedup:        10840
Unique App names:         9659
Duplicate rows removed:   1181
Rows after dedup:         9659


,stage,rows,cols,unique_apps,duplicate_app_rows,missing_rating,note
0,0_raw_loaded,10841,13,9660,1181,1474,Unmodified raw file
1,1_malformed_row_handled,10840,13,9659,1181,1474,1 row dropped: Category unrecoverable after sh...
2,2_deduplicated,9659,13,9659,0,1463,1181 duplicate App rows removed (kept max Revi...


## 4. Target Variable (`Rating`) — Safe Parsing and Validity

`Rating` was already numeric (`float64`) on load, so "parsing" here means
**validating** it: confirming the domain range (1.0–5.0) and deciding how to treat
rows where it is missing.

**Missing-target decision.** `Rating` is the regression target. There is no
statistically defensible way to impute a subjective 1–5 star rating for an app that
has not been rated — doing so would fabricate the exact quantity being predicted and
bias every downstream metric. Rows with a missing `Rating` are therefore **dropped**
from the modelling table. This is a structural requirement of supervised learning
(the label must exist), not a data-dependent statistic, so it introduces no
train/test leakage.


In [8]:
before = df.shape[0]
n_missing_target = df["Rating"].isnull().sum()

df = df[df["Rating"].notnull()].copy()

print(f"Rows with missing Rating (dropped): {n_missing_target}")
print(f"Rows before: {before}  ->  Rows after: {df.shape[0]}")

# Validate the domain range now that the malformed row is fixed and missing
# targets are removed.
print("\nRating range check:", df["Rating"].min(), "to", df["Rating"].max())
assert df["Rating"].between(1.0, 5.0).all(), "Rating out of valid [1,5] range remains!"
print("All remaining Rating values are within the valid [1.0, 5.0] range.")

log_stage("3_target_valid", df, f"{n_missing_target} rows dropped (missing target Rating)")
pd.DataFrame(stage_log)


Rows with missing Rating (dropped): 1463
Rows before: 9659  ->  Rows after: 8196

Rating range check: 1.0 to 5.0
All remaining Rating values are within the valid [1.0, 5.0] range.


,stage,rows,cols,unique_apps,duplicate_app_rows,missing_rating,note
0,0_raw_loaded,10841,13,9660,1181,1474,Unmodified raw file
1,1_malformed_row_handled,10840,13,9659,1181,1474,1 row dropped: Category unrecoverable after sh...
2,2_deduplicated,9659,13,9659,0,1463,1181 duplicate App rows removed (kept max Revi...
3,3_target_valid,8196,13,8196,0,0,1463 rows dropped (missing target Rating)


## 5. Remaining Missing-Value Check (`Type`, `Content Rating`)

The raw audit found exactly one missing `Type` value and one missing `Content
Rating` value. Both are re-checked here now that the malformed row and missing-
target rows have been removed, in case they were already resolved incidentally.


In [9]:
print("Missing Type:", df["Type"].isnull().sum())
print("Missing Content Rating:", df["Content Rating"].isnull().sum())


Missing Type: 0
Missing Content Rating: 0


**Observation.** Both counts are now zero. The row with a missing `Type`
(*"Command & Conquer: Rivals"*, a freshly-listed app with `Installs = '0'`) also had
a missing `Rating`, and the row with a missing `Content Rating` **was** the
malformed row itself (already handled in Section 2). No further rows need to be
dropped for these two columns — this is reported transparently rather than assumed.


In [10]:
before = df.shape[0]
df = df[df["Type"].notnull() & df["Content Rating"].notnull()].copy()
n_dropped = before - df.shape[0]
print(f"Additional rows dropped (Type / Content Rating): {n_dropped}")

log_stage("4_type_contentrating_checked", df,
          f"{n_dropped} additional rows dropped (none expected — verified above)")
pd.DataFrame(stage_log)


Additional rows dropped (Type / Content Rating): 0


,stage,rows,cols,unique_apps,duplicate_app_rows,missing_rating,note
0,0_raw_loaded,10841,13,9660,1181,1474,Unmodified raw file
1,1_malformed_row_handled,10840,13,9659,1181,1474,1 row dropped: Category unrecoverable after sh...
2,2_deduplicated,9659,13,9659,0,1463,1181 duplicate App rows removed (kept max Revi...
3,3_target_valid,8196,13,8196,0,0,1463 rows dropped (missing target Rating)
4,4_type_contentrating_checked,8196,13,8196,0,0,0 additional rows dropped (none expected — ver...


## 6. `Current Ver` and `Android Ver` — Missing Values

`Current Ver` (8 missing) and `Android Ver` (3 missing) are version strings. They
are **not** among the approved predictors (category, reviews, installs, size,
price, app type — Section 1.4 of the report) and are high-cardinality / low-signal
for a rating-prediction task. **Decision: retain the columns with their missing
values left as `NaN`**, for completeness and traceability, and exclude them from
the feature set in `03_feature_engineering.ipynb` rather than impute them here. No
row is dropped for these two columns.


In [11]:
print("Missing Current Ver:", df["Current Ver"].isnull().sum())
print("Missing Android Ver:", df["Android Ver"].isnull().sum())
print("(Both retained as NaN; excluded from the modelling feature set — see notebook 03.)")


Missing Current Ver: 4
Missing Android Ver: 2
(Both retained as NaN; excluded from the modelling feature set — see notebook 03.)


## 7. String Parsing — `Size`, `Installs`, `Price`, `Last Updated`

These four columns are stored as text in the raw file and are parsed into numeric /
datetime forms here. Parsing is a fixed, deterministic function of each cell's own
text — it does **not** compute or fit any statistic from the column's distribution,
so it is leakage-safe to apply to the full dataset (unlike imputation, scaling, or
feature selection, which must wait for the train split).


### 7.1 `Size` → `Size_MB`

Rules: strings ending in `M` are megabytes as written; strings ending in `k`/`K` are
kilobytes and are divided by 1024 to convert to MB; the literal value `'Varies with
device'` means the true size is device-dependent and unknown — it is mapped to
`NaN`, **not** to zero or a guessed value.


In [12]:
def parse_size(s):
    if pd.isnull(s):
        return np.nan
    s = str(s).strip()
    if s == "Varies with device":
        return np.nan
    if s.endswith("M"):
        return float(s[:-1])
    if s.endswith("k") or s.endswith("K"):
        return float(s[:-1]) / 1024.0
    try:
        return float(s)
    except ValueError:
        return np.nan

df["Size_MB"] = df["Size"].apply(parse_size)

n_varies = (df["Size"] == "Varies with device").sum()
n_missing_after = df["Size_MB"].isnull().sum()
print(f"'Varies with device' entries: {n_varies}")
print(f"Size_MB missing after parse:  {n_missing_after}  (should equal the count above)")
assert n_varies == n_missing_after, "Unexpected additional parse failures in Size"
df["Size_MB"].describe()


'Varies with device' entries: 1170
Size_MB missing after parse:  1170  (should equal the count above)


count    7026.000000
mean       21.758067
std        22.727958
min         0.008301
25%         4.900000
50%        13.000000
75%        31.000000
max       100.000000
Name: Size_MB, dtype: float64

**Missing-value decision for `Size_MB`.** These `NaN`s are left as-is in the
clean CSV. Filling them (e.g. with a category-wise median size) is a *learned*
statistic and, per the leakage-prevention rule, must be fit only on the training
split inside the modelling pipeline (`04_modeling_and_tuning.ipynb`), not here.


### 7.2 `Installs` → `Installs_num`

Rule: strip thousands separators (`,`) and the trailing `+`, then cast to integer.
No missing values are expected — every row has some installs string.


In [13]:
def parse_installs(s):
    if pd.isnull(s):
        return np.nan
    s = str(s).replace(",", "").replace("+", "").strip()
    try:
        return int(s)
    except ValueError:
        return np.nan

df["Installs_num"] = df["Installs"].apply(parse_installs)
print("Installs_num missing after parse:", df["Installs_num"].isnull().sum())
df["Installs_num"].describe()


Installs_num missing after parse: 0


count    8.196000e+03
mean     9.189442e+06
std      5.826274e+07
min      1.000000e+00
25%      1.000000e+04
50%      1.000000e+05
75%      1.000000e+06
max      1.000000e+09
Name: Installs_num, dtype: float64

### 7.3 `Price` → `Price_USD`

Rule: strip the leading `$` and cast to float. `'0'` (free apps) parses directly to
`0.0`.


In [14]:
def parse_price(s):
    if pd.isnull(s):
        return np.nan
    s = str(s).replace("$", "").strip()
    try:
        return float(s)
    except ValueError:
        return np.nan

df["Price_USD"] = df["Price"].apply(parse_price)
print("Price_USD missing after parse:", df["Price_USD"].isnull().sum())
print("Paid apps (Price_USD > 0):", (df["Price_USD"] > 0).sum())
df["Price_USD"].describe()


Price_USD missing after parse: 0
Paid apps (Price_USD > 0): 602


count    8196.000000
mean        1.035447
std        16.857244
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       400.000000
Name: Price_USD, dtype: float64

In [15]:
# Sanity check: most expensive apps, to confirm these are real listings and not
# parsing artefacts.
df.sort_values("Price_USD", ascending=False)[["App", "Category", "Price_USD"]].head(10)


,App,Category,Price_USD
4367,I'm Rich - Trump Edition,LIFESTYLE,400.00
5354,I am Rich Plus,FAMILY,399.99
5356,I Am Rich Premium,FINANCE,399.99
5358,I am Rich!,FINANCE,399.99
5359,I am rich(premium),FINANCE,399.99
5362,I Am Rich Pro,FAMILY,399.99
5364,I am rich (Most expensive app),FINANCE,399.99
5351,I am rich,LIFESTYLE,399.99
4362,💎 I'm rich,LIFESTYLE,399.99
5373,I AM RICH PRO PLUS,FINANCE,399.99


**Observation.** The ten most expensive apps are the well-known *"I am Rich"*
family of joke apps priced at $399.99–$400 — these are genuine Play Store listings,
not parsing errors, and are treated as legitimate (if extreme) data points rather
than corrected or removed.


### 7.4 `Last Updated` → `Last_Updated_dt`

Rule: parse with `pandas.to_datetime`. This produces a proper date dtype; deriving
new features from it (e.g. days since update, update year) is left to
`03_feature_engineering.ipynb`, to keep parsing and feature construction separate.


In [16]:
df["Last_Updated_dt"] = pd.to_datetime(df["Last Updated"], errors="coerce")
print("Parse failures:", df["Last_Updated_dt"].isnull().sum())
print("Date range:", df["Last_Updated_dt"].min().date(), "to", df["Last_Updated_dt"].max().date())


Parse failures: 0
Date range: 2010-05-21 to 2018-08-08


In [17]:
log_stage("5_parsed", df, "Size_MB, Installs_num, Price_USD, Last_Updated_dt added; no rows dropped")
pd.DataFrame(stage_log)


,stage,rows,cols,unique_apps,duplicate_app_rows,missing_rating,note
0,0_raw_loaded,10841,13,9660,1181,1474,Unmodified raw file
1,1_malformed_row_handled,10840,13,9659,1181,1474,1 row dropped: Category unrecoverable after sh...
2,2_deduplicated,9659,13,9659,0,1463,1181 duplicate App rows removed (kept max Revi...
3,3_target_valid,8196,13,8196,0,0,1463 rows dropped (missing target Rating)
4,4_type_contentrating_checked,8196,13,8196,0,0,0 additional rows dropped (none expected — ver...
5,5_parsed,8196,17,8196,0,0,"Size_MB, Installs_num, Price_USD, Last_Updated..."


## 8. Outlier Identification

**Method.** The IQR rule is applied to each numeric column: values outside
`[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` (computed on the full cleaned dataset, a fixed
non-learned diagnostic threshold, not a fitted model parameter) are flagged for
inspection. This is a detection pass — the decision on what to do with each flagged
group is made using domain knowledge, not applied automatically.


In [18]:
def iqr_flag(s, name):
    s_valid = s.dropna()
    q1, q3 = s_valid.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((s_valid < lo) | (s_valid > hi)).sum())
    return {
        "column": name,
        "q1": round(q1, 3),
        "q3": round(q3, 3),
        "lower_bound": round(lo, 3),
        "upper_bound": round(hi, 3),
        "n_flagged": n_out,
        "n_valid": len(s_valid),
        "pct_flagged": round(n_out / len(s_valid) * 100, 1),
    }

outlier_report = pd.DataFrame([
    iqr_flag(df["Reviews"], "Reviews"),
    iqr_flag(df["Installs_num"], "Installs_num"),
    iqr_flag(df["Price_USD"], "Price_USD"),
    iqr_flag(df["Size_MB"], "Size_MB"),
    iqr_flag(df["Rating"], "Rating"),
])
outlier_report


,column,q1,q3,lower_bound,upper_bound,n_flagged,n_valid,pct_flagged
0,Reviews,127.0,43976.75,-65647.625,109751.375,1390,8196,17.0
1,Installs_num,10000.0,1000000.00,-1475000.000,2485000.000,1981,8196,24.2
2,Price_USD,0.0,0.00,0.000,0.000,602,8196,7.3
3,Size_MB,4.9,31.00,-34.250,70.150,385,7026,5.5
4,Rating,4.0,4.50,3.250,5.250,491,8196,6.0


**Disposition — decided per variable, not applied automatically:**

| Column | Flagged | Decision | Justification |
|---|---|---|---|
| `Reviews` | ~17% | **Retain; transform, not remove.** Deferred `log1p` transform in `03_feature_engineering.ipynb`. | Review counts are inherently right-skewed across an app store (a handful of viral apps, a long tail of small ones). This is the *real shape of the data*, not measurement error. Removing high-review apps would delete the most successful, best-evidenced ratings in the dataset. |
| `Installs_num` | ~24% | **Retain; transform, not remove.** Deferred `log1p` transform in `03`. | Same reasoning as `Reviews` — install counts span orders of magnitude by nature (1 to 1,000,000,000+); this is expected structure, not error. |
| `Price_USD` | ~7% | **Retain, no transform.** | The IQR bound collapses to `[0, 0]` because the bulk of apps are free — this makes the rule flag *every paid app* as an "outlier," which is really just the Free/Paid structural split, not an anomaly. Kept as-is; `is_paid` and `Price_USD` are both considered in feature engineering. |
| `Size_MB` | ~5.5% | **Retain, no transform.** | Large app sizes (games, media apps) are legitimate; no evidence of corrupted values (all parsed sizes fall in a plausible 0–100MB range for this dataset). |
| `Rating` | ~6% | **Retain — do not remove.** | Flagged values are apps rated below ~3.25, which are genuine low-quality-rated apps. These are essential signal for the regression target itself; "outlier" here only means "statistically less common," not invalid. |

No rows are removed or capped at this stage. The only removals in this notebook are
the single unrecoverable malformed row (Section 2) and rows lacking a usable target
or an unrecoverable minor field (Sections 3–5) — all justified by data validity, not
by statistical extremity.


## 9. Before-vs-After Cleaning Table

Row/column counts and key data-quality statistics captured at every stage of this
notebook.


In [19]:
before_after = pd.DataFrame(stage_log)
before_after


,stage,rows,cols,unique_apps,duplicate_app_rows,missing_rating,note
0,0_raw_loaded,10841,13,9660,1181,1474,Unmodified raw file
1,1_malformed_row_handled,10840,13,9659,1181,1474,1 row dropped: Category unrecoverable after sh...
2,2_deduplicated,9659,13,9659,0,1463,1181 duplicate App rows removed (kept max Revi...
3,3_target_valid,8196,13,8196,0,0,1463 rows dropped (missing target Rating)
4,4_type_contentrating_checked,8196,13,8196,0,0,0 additional rows dropped (none expected — ver...
5,5_parsed,8196,17,8196,0,0,"Size_MB, Installs_num, Price_USD, Last_Updated..."


In [20]:
# Save the stage log alongside the clean data for the report's Section 2.5 table.
before_after.to_csv(f"{PROCESSED_DIR}/cleaning_stage_log.csv", index=False)
print("Saved:", f"{PROCESSED_DIR}/cleaning_stage_log.csv")


Saved: ../data/processed/cleaning_stage_log.csv


## 10. Save Cleaned Dataset

The final cleaned table retains the original text columns (for traceability) plus
the new parsed numeric/datetime columns. No scaler, PCA, feature selector, or
learned imputer has been fit anywhere in this notebook.


In [21]:
final_columns = [
    "App", "Category", "Rating",
    "Reviews", "Size", "Size_MB",
    "Installs", "Installs_num",
    "Type", "Price", "Price_USD",
    "Content Rating", "Genres",
    "Last Updated", "Last_Updated_dt",
    "Current Ver", "Android Ver",
]
df_clean = df[final_columns].copy()
print("Final cleaned shape:", df_clean.shape)
df_clean.head()


Final cleaned shape: (8196, 17)


,App,Category,Rating,Reviews,Size,Size_MB,Installs,Installs_num,Type,Price,Price_USD,Content Rating,Genres,Last Updated,Last_Updated_dt,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,19.0,"10,000+",10000,Free,0,0.0,Everyone,Art & Design,"January 7, 2018",2018-01-07,1.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,8.7,"5,000,000+",5000000,Free,0,0.0,Everyone,Art & Design,"August 1, 2018",2018-08-01,1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,25.0,"50,000,000+",50000000,Free,0,0.0,Teen,Art & Design,"June 8, 2018",2018-06-08,Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,2.8,"100,000+",100000,Free,0,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",2018-06-20,1.1,4.4 and up
5,Paper flowers instructions,ART_AND_DESIGN,4.4,167,5.6M,5.6,"50,000+",50000,Free,0,0.0,Everyone,Art & Design,"March 26, 2017",2017-03-26,1.0,2.3 and up


In [22]:
df_clean.to_csv(CLEAN_OUTPUT_PATH, index=False)
print("Saved:", CLEAN_OUTPUT_PATH)


Saved: ../data/processed/googleplaystore_clean.csv


## 11. Final Verification

A closing set of checks on the saved file: remaining rows/columns, missing values,
duplicates, target validity, dtypes, and that the CSV re-loads cleanly from disk.


In [23]:
print("Remaining rows:", df_clean.shape[0])
print("Remaining columns:", df_clean.shape[1])


Remaining rows: 8196
Remaining columns: 17


In [24]:
print("Missing values per column:")
df_clean.isnull().sum()


Missing values per column:


App                   0
Category              0
Rating                0
Reviews               0
Size                  0
Size_MB            1170
Installs              0
Installs_num          0
Type                  0
Price                 0
Price_USD             0
Content Rating        0
Genres                0
Last Updated          0
Last_Updated_dt       0
Current Ver           4
Android Ver           2
dtype: int64

In [25]:
print("Duplicate App names remaining:", df_clean.duplicated(subset=['App']).sum())
print("Fully duplicated rows remaining:", df_clean.duplicated().sum())


Duplicate App names remaining: 0


Fully duplicated rows remaining: 0


In [26]:
print("Target (Rating) validity:")
print("  missing:", df_clean["Rating"].isnull().sum())
print("  min:", df_clean["Rating"].min(), " max:", df_clean["Rating"].max())
assert df_clean["Rating"].notnull().all()
assert df_clean["Rating"].between(1.0, 5.0).all()
print("  OK — Rating is fully populated and within [1.0, 5.0] for every row.")


Target (Rating) validity:
  missing: 0
  min: 1.0  max: 5.0
  OK — Rating is fully populated and within [1.0, 5.0] for every row.


In [27]:
df_clean.dtypes


App                           str
Category                      str
Rating                    float64
Reviews                     int64
Size                          str
Size_MB                   float64
Installs                      str
Installs_num                int64
Type                          str
Price                         str
Price_USD                 float64
Content Rating                str
Genres                        str
Last Updated                  str
Last_Updated_dt    datetime64[us]
Current Ver                   str
Android Ver                   str
dtype: object

In [28]:
# Reload from disk exactly as saved, to confirm the CSV is loadable and consistent.
reloaded = pd.read_csv(CLEAN_OUTPUT_PATH)
print("Reloaded shape:", reloaded.shape)
assert reloaded.shape == df_clean.shape
assert list(reloaded.columns) == list(df_clean.columns)
assert reloaded["Rating"].between(1.0, 5.0).all()
assert reloaded.duplicated(subset=["App"]).sum() == 0
print("Reload check passed: shape, columns, target range, and uniqueness all match.")


Reloaded shape: (8196, 17)
Reload check passed: shape, columns, target range, and uniqueness all match.


## 12. Summary of Cleaning Decisions

| Issue | Action | Rows affected |
|---|---|---|
| Malformed shifted row (idx 10472) | Corrected recoverable fields; dropped (unrecoverable `Category`) | 1 dropped |
| Duplicate `App` rows | Deduplicated on `App`, kept max-`Reviews` row per app | 1,181 dropped |
| Missing target (`Rating`) | Dropped (target cannot be imputed for supervised learning) | 1,463 dropped |
| Missing `Type` / `Content Rating` | Verified zero remaining after prior steps | 0 dropped |
| Missing `Current Ver` / `Android Ver` | Retained as `NaN`; excluded from modelling features | 0 dropped |
| `Size` = `'Varies with device'` | Parsed to `NaN`; imputation deferred to train-only pipeline | 0 dropped (values left missing) |
| `Installs`, `Price`, `Reviews` | Parsed to numeric; no missing values produced | 0 dropped |
| `Last Updated` | Parsed to datetime; 0 parse failures | 0 dropped |
| Statistical outliers (IQR) in `Reviews`, `Installs`, `Price`, `Size` | Retained; skewed count variables deferred to `log1p` transform in feature engineering | 0 dropped |
| Statistical outliers (IQR) in `Rating` (< ~3.25) | Retained — genuine low ratings, essential target signal | 0 dropped |

**Net result:** 10,841 raw rows → **8,196 clean rows**, 13 raw columns → **17
columns** (13 original + 4 parsed). No leakage-sensitive transformation (scaler,
PCA, feature selector, or learned imputer) was fit anywhere in this notebook; those
steps are deferred to `04_modeling_and_tuning.ipynb`, fit on the training split only.
